In [1]:
"""
This code loads the ETTm2 time-series dataset, selects seven transformer-related
features, and converts them into NumPy arrays for model training and testing.

It defines a sliding-window setup where each sample contains:
- m observed time steps used as input,
- k future time steps used to determine the target label.

The dataset is sorted by date, split chronologically into training and testing
regions, and overlapping window start indices are generated using the chosen
stride. Assertions ensure that training windows remain entirely before the
train/test cutoff and test windows begin only after the cutoff.

Each window is assigned a binary label based on whether the
future value of the OT feature exceeds the threshold tau. The build_arrays
function then constructs separate arrays for observed inputs, future values,
and labels for both training and testing sets.

"""
import numpy as np
import pandas as pd
import pathlib
import os
import random
import torch
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
from MyModels import ModernTCNClassifier,   ModernTCNForecastThenClassifyOfficial, BayesianModernTCNForecastThenClassifyJoint, RocketMultivariateLengths
from tensorflow.keras.utils import to_categorical 
from sklearn.linear_model import RidgeClassifierCV
CSV_PATH    = "~/data/ETDataset/ETT-small/ETTm2.csv"
m           = 4      
k           = 48     
tau         = 30     
train_frac  = 0.8   
stride      = 1      
COLS        = ['HUFL','HULL','MUFL','MULL','LUFL','LULL','OT']
df = (pd.read_csv(pathlib.Path(CSV_PATH).expanduser(),
                  parse_dates=['date'])
        .sort_values('date'))
X = df[COLS].to_numpy(dtype=np.float32)   
T = len(X)
window = m + k
cutoff_idx = int(np.floor(train_frac * T))
train_starts = np.arange(0, cutoff_idx - window + 1, stride)
test_starts  = np.arange(cutoff_idx, T         - window + 1, stride)
assert np.all(train_starts + window <= cutoff_idx)
assert np.all(test_starts >= cutoff_idx)
print(f"Total windows          : {T - window + 1}")
print(f"Train windows (overlap): {len(train_starts)}")
print(f"Test  windows (overlap): {len(test_starts)}")
def early_warning_label(win, tau=tau):
    ot_future = win[m:, 6]
    return int(ot_future.max() > tau)
def build_arrays(starts):
    n = len(starts)
    x_obs = np.empty((n, m, 7), dtype=np.float32)
    x_fut = np.empty((n, k, 7), dtype=np.float32)
    y     = np.empty((n,),    dtype=np.int64)
    for idx, i in enumerate(starts):
        w = X[i : i + window]
        x_obs[idx] = w[:m]
        x_fut[idx] = w[m:]
        y[idx]     = early_warning_label(w)
    return x_obs, x_fut, y
x_obs_train, x_fut_train, y_train = build_arrays(train_starts)
x_obs_test,  x_fut_test,  y_test  = build_arrays(test_starts)
print("\nShapes:")
print(" x_obs_train:", x_obs_train.shape)
print(" x_fut_train:", x_fut_train.shape)
print(" y_train    :", y_train.shape)
print(" x_obs_test :", x_obs_test.shape)
print(" x_fut_test :", x_fut_test.shape)
print(" y_test     :", y_test.shape)

/opt/anaconda3/lib/python3.12/site-packages/aeon/base/__init__.py:24: FutureWarning: The aeon package will soon be releasing v1.0.0 with the removal of legacy modules and interfaces such as BaseTransformer and BaseForecaster. This will contain breaking changes. See aeon-toolkit.org for more information. Set aeon.AEON_DEPRECATION_WARNING or the AEON_DEPRECATION_WARNING environmental variable to 'False' to disable this warning.
  warnings.warn(
/var/folders/t0/_3__vd353d18by07shp_y80m0000gn/T/ipykernel_904/3730560200.py:18: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = (pd.read_csv(pathlib.Path(CSV_PATH).expanduser(),


Total windows          : 69629
Train windows (overlap): 55693
Test  windows (overlap): 13885

Shapes:
 x_obs_train: (55693, 4, 7)
 x_fut_train: (55693, 48, 7)
 y_train    : (55693,)
 x_obs_test : (13885, 4, 7)
 x_fut_test : (13885, 48, 7)
 y_test     : (13885,)


In [2]:
"""
This code prints the shapes of the prepared training and testing arrays to
confirm that the observed inputs, future sequences, and label vectors have
been created with the expected dimensions.

It then checks the class distribution of the binary labels in both the training
and testing sets. For each unique class label, it counts how many examples are
present and calculates the percentage of the dataset belonging to that class.
"""
print("Shapes:")
print(" x_obs_train:", x_obs_train.shape)
print(" x_fut_train:", x_fut_train.shape)
print(" y_train    :", y_train.shape)
print(" x_obs_test :", x_obs_test.shape)
print(" x_fut_test :", x_fut_test.shape)
print(" y_test     :", y_test.shape)
unique_train, counts_train = np.unique(y_train, return_counts=True)
for cls, cnt in zip(unique_train, counts_train):
    pct = cnt / len(y_train) * 100
    print(f"Train class {cls}: {cnt} examples ({pct:.1f}%)")
unique_test, counts_test = np.unique(y_test, return_counts=True)
for cls, cnt in zip(unique_test, counts_test):
    pct = cnt / len(y_test) * 100
    print(f" Test class {cls}: {cnt} examples ({pct:.1f}%)")

Shapes:
 x_obs_train: (55693, 4, 7)
 x_fut_train: (55693, 48, 7)
 y_train    : (55693,)
 x_obs_test : (13885, 4, 7)
 x_fut_test : (13885, 48, 7)
 y_test     : (13885,)
Train class 0: 27623 examples (49.6%)
Train class 1: 28070 examples (50.4%)
 Test class 0: 6046 examples (43.5%)
 Test class 1: 7839 examples (56.5%)


In [3]:
"""
This code defines a helper function to balance a binary classification dataset
by randomly downsampling the larger class to match the size of the smaller
class.

It identifies the indices of class 0 and class 1 examples, finds the minority
class size, and randomly selects that same number of samples from each class
without replacement. The selected indices are then combined, shuffled, and used
to return balanced versions of the observed inputs, future sequences, and
labels.

The function is applied to the training data using a fixed random seed so that
the downsampling process is reproducible. Finally, Counter is used to print the
updated class counts in y_train, confirming that the training set is now
balanced.
"""
def balance_downsample(x_obs, x_fut, y, random_state=None):
    rng = np.random.default_rng(random_state)
    idx0 = np.where(y == 0)[0]
    idx1 = np.where(y == 1)[0]
    n_min = min(len(idx0), len(idx1))
    sel0 = rng.choice(idx0, size=n_min, replace=False)
    sel1 = rng.choice(idx1, size=n_min, replace=False)
    sel = np.concatenate([sel0, sel1])
    rng.shuffle(sel)
    return x_obs[sel], x_fut[sel], y[sel]
x_obs_train, x_fut_train, y_train = balance_downsample(
    x_obs_train, x_fut_train, y_train, random_state=42
)
from collections import Counter
print(Counter(y_train))

Counter({1: 27623, 0: 27623})


In [4]:
"""
This code assigns the prepared training and testing arrays to clearer variable
names that distinguish between observed input sequences, future forecast
sequences, and target labels.

It then selects the best available compute device for PyTorch, prioritising
Apple Silicon MPS if available, then CUDA for an NVIDIA GPU, and finally falling
back to the CPU if no supported accelerator is detected. 

Finally, the to_int helper function ensures that label arrays are represented
as integer class labels. If the labels are one-hot encoded, it converts them
using argmax; otherwise, it casts the existing labels directly to integers.
"""
X_observed_train =  x_obs_train
X_forecast_train =  x_fut_train
y_train =  y_train
X_observed_test =  x_obs_test
X_forecast_test =   x_fut_test 
y_test = y_test
device = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(f"Using device: {device}")
def to_int(y):
    y = np.asarray(y)
    return np.argmax(y, axis=1) if (y.ndim == 2 and y.shape[1] > 1) else y.astype(int)  

Using device: mps


In [5]:
"""
This code trains and evaluates a ModernTCNClassifier multiple times using
different random seeds to assess how stable the model performance is across
repeated runs.

The set_seed function configures Python, NumPy, and PyTorch random seeds so
that each run is as reproducible as possible. It also enables deterministic
PyTorch behaviour and disables cuDNN benchmarking to reduce randomness during
GPU-based training.

For each seed, the code:
- sets the random seed,
- creates a new ModernTCNClassifier with fixed training hyperparameters,
- trains the model on the observed training sequences,
- predicts class labels and class probabilities on the observed test sequences.
Performance metrics are then computed and printed. 

"""
SEED_BASE = 0  
def set_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8" 
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
results = []  
for seed in range(5):  
    print("\n" + "="*80)
    print(f"Training ModernTCNClassifier - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = ModernTCNClassifier(
        max_epochs=50,       
        batch_size=512,      
        lr=1e-3,
        weight_decay=1e-4,
        device=device,       
        verbose=True,
    )
    clf.fit(X_observed_train, y_train)
    y_pred = clf.predict(X_observed_test)          
    proba  = clf.predict_proba(X_observed_test)    
    y_true = y_test.astype(int)
    print("\nClassification report (seed {}):".format(seed))
    print(classification_report(y_true, y_pred, digits=4))
    if proba.shape[1] >= 2:
        y_score = proba[:, 1].ravel()  
        auroc = roc_auc_score(y_true, y_score)
        auprc = average_precision_score(y_true, y_score)
        y_score_min = proba[:, 0].ravel() 
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (seed {seed}): {auroc:.4f}")
        print(f"AUPRC (seed {seed}): {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append({"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0":auprc0})
    else:
        print("Warning: only one probability column, skipping AUROC/AUPRC.")
        results.append({"seed": seed, "auroc": np.nan, "auprc": np.nan,"auprc0":auprc0 })
print("\n" + "#"*80)
print("Summary over seeds")
print("#"*80)
for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}"
        f"AUPRC0={r['auprc0']:.4f}"
    )


Training ModernTCNClassifier - seed 0
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|████████████████| 108/108 [00:03<00:00, 35.71it/s]



Classification report (seed 0):
              precision    recall  f1-score   support

           0     0.8396    0.7878    0.8129      6046
           1     0.8438    0.8839    0.8634      7839

    accuracy                         0.8421     13885
   macro avg     0.8417    0.8359    0.8381     13885
weighted avg     0.8419    0.8421    0.8414     13885

AUROC (seed 0): 0.9308
AUPRC (seed 0): 0.9566
AUPRC class 0 (seed 0): 0.8933

Training ModernTCNClassifier - seed 1
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|████████████████| 108/108 [00:02<00:00, 36.22it/s]



Classification report (seed 1):
              precision    recall  f1-score   support

           0     0.8711    0.8023    0.8353      6046
           1     0.8563    0.9084    0.8816      7839

    accuracy                         0.8622     13885
   macro avg     0.8637    0.8554    0.8584     13885
weighted avg     0.8627    0.8622    0.8614     13885

AUROC (seed 1): 0.9482
AUPRC (seed 1): 0.9646
AUPRC class 0 (seed 1): 0.9273

Training ModernTCNClassifier - seed 2
[ModernTCN] Using device: mps


...


Classification report (seed 2):
              precision    recall  f1-score   support

           0     0.8383    0.8435    0.8409      6046
           1     0.8787    0.8745    0.8766      7839

    accuracy                         0.8610     13885
   macro avg     0.8585    0.8590    0.8587     13885
weighted avg     0.8611    0.8610    0.8610     13885

AUROC (seed 2): 0.9492
AUPRC (seed 2): 0.9647
AUPRC class 0 (seed 2): 0.9335

Training ModernTCNClassifier - seed 3
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|████████████████| 108/108 [00:03<00:00, 35.76it/s]



Classification report (seed 3):
              precision    recall  f1-score   support

           0     0.8705    0.8194    0.8442      6046
           1     0.8667    0.9060    0.8859      7839

    accuracy                         0.8683     13885
   macro avg     0.8686    0.8627    0.8650     13885
weighted avg     0.8684    0.8683    0.8677     13885

AUROC (seed 3): 0.9538
AUPRC (seed 3): 0.9679
AUPRC class 0 (seed 3): 0.9378

Training ModernTCNClassifier - seed 4
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|████████████████| 108/108 [00:02<00:00, 36.05it/s]



Classification report (seed 4):
              precision    recall  f1-score   support

           0     0.8658    0.8720    0.8689      6046
           1     0.9007    0.8958    0.8982      7839

    accuracy                         0.8854     13885
   macro avg     0.8833    0.8839    0.8836     13885
weighted avg     0.8855    0.8854    0.8855     13885

AUROC (seed 4): 0.9593
AUPRC (seed 4): 0.9711
AUPRC class 0 (seed 4): 0.9458

################################################################################
Summary over seeds
################################################################################
Seed 0: AUROC=0.9308  AUPRC=0.9566AUPRC0=0.8933
Seed 1: AUROC=0.9482  AUPRC=0.9646AUPRC0=0.9273
Seed 2: AUROC=0.9492  AUPRC=0.9647AUPRC0=0.9335
Seed 3: AUROC=0.9538  AUPRC=0.9679AUPRC0=0.9378
Seed 4: AUROC=0.9593  AUPRC=0.9711AUPRC0=0.9458


In [6]:
"""
This code trains and evaluates the ModernTCNForecastThenClassify model
across five different random seeds to assess how stable its performance is
across repeated runs.

For each seed, the code resets the random state for reproducibility, creates a
new forecast-then-classify model, and provides it with the future training
sequences needed for the forecasting stage. The model is then trained using the
observed training sequences and their binary class labels.

After training, the model predicts class probabilities for the observed test
sequences. These probabilities are converted into predicted class labels using
argmax. Performance metrics are then computed and printed. 
"""
results = []  
for seed in range(5): 
    print("\n" + "="*80)
    print(f"Training ModernTCNForecastThenClassifyOfficial - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = ModernTCNForecastThenClassifyOfficial(
        X_forecast_train=X_forecast_train,
        max_epochs_forecast=20,
        max_epochs_class=50,
        batch_size=512,          
        lr_forecast=1e-3,
        lr_class=1e-3,
        weight_decay=1e-4,
        device=device,
        verbose=True,
    )
    clf.fit(X_observed_train.astype(np.float32), y_train.astype(int))
    proba = clf.predict_proba(X_observed_test.astype(np.float32))   
    y_pred = np.argmax(proba, axis=1)
    y_true = y_test.astype(int)
    print("\nClassification report (seed {}):".format(seed))
    print(classification_report(y_true, y_pred, digits=4))
    if proba.shape[1] >= 2:
        y_score = proba[:, 1].ravel()  
        auroc = roc_auc_score(y_true, y_score)
        auprc = average_precision_score(y_true, y_score)
        y_score_min = proba[:, 0].ravel() 
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (seed {seed}): {auroc:.4f}")
        print(f"AUPRC (seed {seed}): {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append({"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0":auprc0})
    else:
        print("Warning: only one probability column, skipping AUROC/AUPRC.")
        results.append({"seed": seed, "auroc": np.nan, "auprc": np.nan,"auprc0":np.nan })
print("\n" + "#"*80)
print("Summary over seeds")
print("#"*80)
for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}"
        f"AUPRC0={r['auprc0']:.4f}"
    )


Training ModernTCNForecastThenClassifyOfficial - seed 0
[TwoStage-ModernTCN] Using device: mps


[ModernTCN-forecast] Epoch 20/20: 100%|███████| 108/108 [00:04<00:00, 25.84it/s]


[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|████████████████| 108/108 [00:03<00:00, 30.27it/s]



Classification report (seed 0):
              precision    recall  f1-score   support

           0     0.8476    0.8492    0.8484      6046
           1     0.8835    0.8823    0.8829      7839

    accuracy                         0.8678     13885
   macro avg     0.8656    0.8657    0.8656     13885
weighted avg     0.8679    0.8678    0.8679     13885

AUROC (seed 0): 0.9427
AUPRC (seed 0): 0.9628
AUPRC class 0 (seed 0): 0.9158

Training ModernTCNForecastThenClassifyOfficial - seed 1
[TwoStage-ModernTCN] Using device: mps


...


Classification report (seed 3):
              precision    recall  f1-score   support

           0     0.8485    0.8354    0.8419      6046
           1     0.8746    0.8849    0.8797      7839

    accuracy                         0.8634     13885
   macro avg     0.8615    0.8602    0.8608     13885
weighted avg     0.8632    0.8634    0.8633     13885

AUROC (seed 3): 0.9485
AUPRC (seed 3): 0.9640
AUPRC class 0 (seed 3): 0.9315

Training ModernTCNForecastThenClassifyOfficial - seed 4
[TwoStage-ModernTCN] Using device: mps


[ModernTCN-forecast] Epoch 20/20: 100%|███████| 108/108 [00:04<00:00, 26.10it/s]


[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|████████████████| 108/108 [00:03<00:00, 31.45it/s]



Classification report (seed 4):
              precision    recall  f1-score   support

           0     0.8045    0.9153    0.8563      6046
           1     0.9269    0.8284    0.8749      7839

    accuracy                         0.8663     13885
   macro avg     0.8657    0.8719    0.8656     13885
weighted avg     0.8736    0.8663    0.8668     13885

AUROC (seed 4): 0.9494
AUPRC (seed 4): 0.9646
AUPRC class 0 (seed 4): 0.9339

################################################################################
Summary over seeds
################################################################################
Seed 0: AUROC=0.9427  AUPRC=0.9628AUPRC0=0.9158
Seed 1: AUROC=0.9400  AUPRC=0.9603AUPRC0=0.9142
Seed 2: AUROC=0.9523  AUPRC=0.9671AUPRC0=0.9357
Seed 3: AUROC=0.9485  AUPRC=0.9640AUPRC0=0.9315
Seed 4: AUROC=0.9494  AUPRC=0.9646AUPRC0=0.9339


In [8]:
"""
This code trains and evaluates the ForeClassNet model across five different 
random seeds, using Monte Carlo dropout to estimate prediction probabilities.

The MC_SAMPLES value controls how many stochastic forward passes are used during
Bayesian prediction. For each seed, the code resets the random state for
reproducibility, creates a new ForeClassNet model, and trains
it using the observed training sequences, future forecast training sequences,
and binary class labels.

After training, predict_proba_bayesian is called on the observed test sequences.
This returns the mean predicted class probabilities across the Monte Carlo
samples.
Performance metrics are then computed and printed. 
"""
MC_SAMPLES = 100  
results = []
for seed in range(5):  
    print("\n" + "="*80)
    print(f"Training BayesianModernTCNForecastThenClassifyJoint - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = BayesianModernTCNForecastThenClassifyJoint(
        X_forecast_train=X_forecast_train.astype(np.float32),
        max_epochs_forecast=20,
        max_epochs_class=50,
        batch_size=512,          
        lr_forecast=1e-3,
        lr_class=1e-3,
        weight_decay=1e-4,
        device=device,
        verbose=True,
        l=1e-2,
        tau=1.0,
    )
    clf.fit(X_observed_train.astype(np.float32), y_train.astype(int))
    mean_proba, std_proba = clf.predict_proba_bayesian(
        X_observed_test.astype(np.float32),
        n_samples=MC_SAMPLES,
    )  
    y_true = y_test.astype(int)
    y_pred = np.argmax(mean_proba, axis=1)
    print("\nClassification Report (MC dropout, {} samples, seed {}):".format(MC_SAMPLES, seed))
    print(classification_report(y_true, y_pred, digits=4))
    print("-"*72)
    if mean_proba.shape[1] >= 2:
        y_score = mean_proba[:, 1].ravel()
        auroc = roc_auc_score(y_true, y_score)
        auprc = average_precision_score(y_true, y_score)
        y_score_min = mean_proba[:, 0].ravel() 
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (MC mean over {MC_SAMPLES} samples, seed {seed}) : {auroc:.4f}")
        print(f"AUPRC (MC mean over {MC_SAMPLES} samples, seed {seed}) : {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append({"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0":auprc0})
    else:
        print(f"Warning: only one probability column (shape={mean_proba.shape}); skipping AUROC/AUPRC.")
        results.append({"seed": seed, "auroc": np.nan, "auprc": np.nan, "auprc0":np.nan})
print("\n" + "#"*80)
print(f"Summary over seeds (BayesianModernTCNForecastThenClassifyJoint, MC={MC_SAMPLES})")
print("#"*80)
for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}"
        f"AUPRC0={r['auprc0']:.4f}"
    )


Training BayesianModernTCNForecastThenClassifyJoint - seed 0
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|██| 108/108 [00:27<00:00,  3.99it/s]



Classification Report (MC dropout, 100 samples, seed 0):
              precision    recall  f1-score   support

           0     0.8728    0.8202    0.8457      6046
           1     0.8675    0.9078    0.8872      7839

    accuracy                         0.8696     13885
   macro avg     0.8701    0.8640    0.8664     13885
weighted avg     0.8698    0.8696    0.8691     13885

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 0) : 0.9572
AUPRC (MC mean over 100 samples, seed 0) : 0.9699
AUPRC class 0 (seed 0): 0.9438

Training BayesianModernTCNForecastThenClassifyJoint - seed 1
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|██| 108/108 [00:25<00:00,  4.19it/s]



Classification Report (MC dropout, 100 samples, seed 1):
              precision    recall  f1-score   support

           0     0.8922    0.8052    0.8465      6046
           1     0.8602    0.9250    0.8914      7839

    accuracy                         0.8728     13885
   macro avg     0.8762    0.8651    0.8690     13885
weighted avg     0.8742    0.8728    0.8719     13885

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 1) : 0.9569
AUPRC (MC mean over 100 samples, seed 1) : 0.9697
AUPRC class 0 (seed 1): 0.9433

Training BayesianModernTCNForecastThenClassifyJoint - seed 2
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


...


Classification Report (MC dropout, 100 samples, seed 2):
              precision    recall  f1-score   support

           0     0.8549    0.8377    0.8462      6046
           1     0.8768    0.8903    0.8835      7839

    accuracy                         0.8674     13885
   macro avg     0.8658    0.8640    0.8648     13885
weighted avg     0.8672    0.8674    0.8672     13885

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 2) : 0.9575
AUPRC (MC mean over 100 samples, seed 2) : 0.9701
AUPRC class 0 (seed 2): 0.9444

Training BayesianModernTCNForecastThenClassifyJoint - seed 3
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|██| 108/108 [00:25<00:00,  4.17it/s]



Classification Report (MC dropout, 100 samples, seed 3):
              precision    recall  f1-score   support

           0     0.8704    0.8263    0.8478      6046
           1     0.8711    0.9051    0.8878      7839

    accuracy                         0.8708     13885
   macro avg     0.8707    0.8657    0.8678     13885
weighted avg     0.8708    0.8708    0.8704     13885

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 3) : 0.9582
AUPRC (MC mean over 100 samples, seed 3) : 0.9707
AUPRC class 0 (seed 3): 0.9448

Training BayesianModernTCNForecastThenClassifyJoint - seed 4
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|██| 108/108 [00:25<00:00,  4.17it/s]



Classification Report (MC dropout, 100 samples, seed 4):
              precision    recall  f1-score   support

           0     0.8598    0.8189    0.8389      6046
           1     0.8653    0.8971    0.8809      7839

    accuracy                         0.8630     13885
   macro avg     0.8626    0.8580    0.8599     13885
weighted avg     0.8629    0.8630    0.8626     13885

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 4) : 0.9557
AUPRC (MC mean over 100 samples, seed 4) : 0.9690
AUPRC class 0 (seed 4): 0.9418

################################################################################
Summary over seeds (BayesianModernTCNForecastThenClassifyJoint, MC=100)
################################################################################
Seed 0: AUROC=0.9572  AUPRC=0.9699AUPRC0=0.9438
Seed 1: AUROC=0.9569  AUPRC=0.9697AUPRC0=0.9433
Seed 2: AUROC=0.9575  AUPRC=0.9701AUPRC0=0.9444
Seed 3: AUROC=0.9582  AUPRC=0.97

In [12]:
"""
This code trains and evaluates a ROCKET classifier across five different random seeds.

The prepare_labels function standardises the target labels into a one-dimensional
integer array. It supports labels that are already one-dimensional, labels stored
as a single-column two-dimensional array, and one-hot encoded labels. If the
label array has an unsupported shape, it raises an error.

For each seed, the code resets the random state for reproducibility and creates
a RocketMultivariateLengths transformer. The transformer generates a large set
of random convolutional features from the observed multivariate time-series
inputs using several kernel lengths.

The ROCKET transformer is fitted on the observed training data, then used to
convert both training and testing sequences into feature matrices. A
RidgeClassifierCV model is then trained on the transformed training features,
using cross-validation to choose the best regularisation strength from a range
of alpha values.

After training, the classifier predicts test labels and produces decision scores
for the test set. Performance metrics are then computed and printed. 
"""
def prepare_labels(y):
    y = np.asarray(y)
    if y.ndim == 1:
        return y.astype(int)
    if y.ndim == 2:
        if y.shape[1] == 1:
            return y.ravel().astype(int)
        return y.argmax(axis=1).astype(int)
    raise ValueError(f"Unsupported y shape {y.shape}")
results = []
for seed in range(5):  
    print("\n" + "="*80)
    print(f"Training ROCKET + RidgeClassifierCV - seed {seed}")
    print("="*80)

    set_seed(seed)
    rocket = RocketMultivariateLengths(
        n_kernels=10000,        
        lengths=(2, 3, 4),
        random_state= seed,
    )
    rocket.fit(X_observed_train.astype(np.float32))
    X_train_feat = rocket.transform(X_observed_train.astype(np.float32))
    X_test_feat  = rocket.transform(X_observed_test.astype(np.float32))
    y_train_labels = prepare_labels(y_train)
    y_test_labels  = prepare_labels(y_test)
    clf = RidgeClassifierCV(alphas=np.logspace(-3, 3, 10))
    clf.fit(X_train_feat, y_train_labels)
    y_pred = clf.predict(X_test_feat)
    decision_scores = clf.decision_function(X_test_feat)  
    print("\nClassification report (seed {}):".format(seed))
    print(classification_report(y_test_labels, y_pred, digits=4))
    auroc = roc_auc_score(y_test_labels, decision_scores)
    auprc = average_precision_score(y_test_labels, decision_scores)
    auprc_class0 = average_precision_score(y_test_labels == 0, -decision_scores)
    print(f"AUROC (seed {seed}): {auroc:.4f}")
    print(f"AUPRC (seed {seed}): {auprc:.4f}")
    print(f"AUPRC0 (seed {seed}): {auprc_class0:.4f}")
    results.append({"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0":auprc_class0})
print("\n" + "#"*80)
print("Summary over seeds (ROCKET + RidgeClassifierCV)")
print("#"*80)
for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}"
        f"AUPRC0={r['auprc0']:.4f}"
    )


Training ROCKET + RidgeClassifierCV - seed 0

Classification report (seed 0):
              precision    recall  f1-score   support

           0     0.8545    0.8308    0.8425      6046
           1     0.8722    0.8909    0.8815      7839

    accuracy                         0.8647     13885
   macro avg     0.8634    0.8609    0.8620     13885
weighted avg     0.8645    0.8647    0.8645     13885

AUROC (seed 0): 0.9325
AUPRC (seed 0): 0.9402
AUPRC0 (seed 0): 0.9124

Training ROCKET + RidgeClassifierCV - seed 1

Classification report (seed 1):
              precision    recall  f1-score   support

           0     0.8438    0.8452    0.8445      6046
           1     0.8804    0.8793    0.8799      7839

    accuracy                         0.8645     13885
   macro avg     0.8621    0.8623    0.8622     13885
weighted avg     0.8645    0.8645    0.8645     13885

AUROC (seed 1): 0.9252
AUPRC (seed 1): 0.9213
AUPRC0 (seed 1): 0.9126

Training ROCKET + RidgeClassifierCV - seed 2

C